In [1]:
from transformers import pipeline

# Summarizer


In [2]:
summarizer = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
import json

with open('../data/top_headlines_us_context.json', 'r') as f:
    headlines_with_context = json.load(f)


In [14]:
from transformers import logging

logging.set_verbosity_error()

In [ ]:
from datasets import Dataset
from tqdm import tqdm
import re

PROMPT_TEMPLATE = (
    "Summarize the following news article in 2-3 sentences.\n"
    "Article:\n{context}\nTitle: {title}\nDescription: {description}\nContent: {content}\nSummary:"
)

def extract_summary(text):
    # Remove everything before 'Summary:' if present
    match = re.search(r"Summary:(.*)", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

texts = [
    PROMPT_TEMPLATE.format(
        context=str(article.get('context', '')),
        title=str(article.get('title', '')),
        description=str(article.get('description', '')),
        content=str(article.get('content', ''))
    )
    for article in headlines_with_context
]
sources = [article.get('source', {}) for article in headlines_with_context]
urls = [article.get('url', '') for article in headlines_with_context]

dataset = Dataset.from_dict({"text": texts, "source": sources, "url": urls})

batch_size = 8
summaries = []
for i in tqdm(range(0, len(dataset), batch_size), desc="Batch summarizing"):
    batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
    batch_texts = list(batch['text'])
    batch_outputs = summarizer(batch_texts, max_new_tokens=128, max_length=None)
    for output, source_obj in zip(batch_outputs, batch['source']):
        if isinstance(output, list):
            summary_text = output[0]['generated_text'] if output and 'generated_text' in output[0] else str(output)
        else:
            summary_text = output.get('generated_text', str(output))
        clean_summary = extract_summary(summary_text)
        summaries.append({
            "Summary": clean_summary,
            "source": source_obj
        })

Batch summarizing:   0%|          | 0/3 [00:00<?, ?it/s]

Batch summarizing: 100%|██████████| 3/3 [01:14<00:00, 24.76s/it]


In [16]:
for s in summaries:
    print(f"Summary:\n{s['Summary']}\nsource: {s['source']}\n")

Summary:
Summarize the following news article in 2-3 sentences.
Article:
'RHOA's Kim Zolciak to Be Grilled by BF’s Estranged Wife in Divorce War - TMZ

'Real Housewives of Atlanta' alum Kim Zolciak agreed to be questioned under oath by lawyers representing her boyfriend Kyle Mowitz’s estranged wife, Jillian Green ... TMZ has learned.

'Real Housewives of Atlanta' alum Kim Zolciak agreed to be questioned under oath by lawyers representing her boyfriend Kyle Mowitzs estranged wife, Jillian Green ... TMZ has learned.
According to co… [+1142 chars]

'Real Housewives of Atlanta' alum Kim Zolciak agreed to be questioned under oath by lawyers representing her boyfriend Kyle Mowitz’s estranged wife, Jillian Green ... TMZ has learned. According to court docs obtained by TMZ, Kim and Jillian reached a deal where the reality star will show up and answer questions. According to the agreement, the transcript for Kim’s deposition will be kept private and won't be shared with any third parties. As TM

# Parse the context added headlines into objects

